<div style="text-align: center; margin-bottom: 20px;">
    <img src="logo/Logo_Facyt.svg.png" width="180" style="display: block; margin-left: auto; margin-right: auto; margin-bottom: 10px;">
    <h2 style="color: #1B365D; margin-bottom: 2px;">UNIVERSIDAD DE CARABOBO</h2>
    <h3 style="color: #2B5B84; margin-top: 0px;">FACULTAD EXPERIMENTAL DE CIENCIAS Y TECNOLOGÍA (FACYT)</h3>
    <h4 style="color: #4A777A;">DEPARTAMENTO DE COMPUTACIÓN</h4>
    <p><strong>Course:</strong> Machine Learning (Elective 2026) | <strong>Professor:</strong> Álvaro Espinoza</p>
    <p><strong>Authors:</strong> Anthony Quero & Luigi Quero</p>
    <hr style="border: 1px solid #2B5B84; margin: 15px 0;">
    <h1 style="color: #D9534F; margin-bottom: 5px;">Assignment 2: Data Preparation & Feature Engineering</h1>
    <h3 style="color: #333333;">Project: Pokémon Battle Winner Predictor</h3>
</div>

# 1. Purpose and Problem Context

In this notebook, we transform the empirical findings from our exploratory analysis (`01_eda_pokemon.ipynb`) into a **reproducible, leak-free data preparation and feature engineering pipeline**. Following the course design principles, we do not train, compare, or tune candidate models here: model evaluation belongs strictly to the upcoming Assignment 3 (Modeling) notebook.

| Element | Formal Definition in this Project |
|---|---|
| **Unit of observation** | A single historical combat between `First_pokemon` (Position 1) and `Second_pokemon` (Position 2). |
| **Target variable** | `Target_First_Wins` $\in \{0, 1\}$, where $1$ denotes `Winner == First_pokemon` and $0$ denotes `Winner == Second_pokemon` (binary classification). |
| **Prediction meaning** | The estimated probability that the Pokémon entering in Position 1 wins a **previously unobserved combat** between two known catalogue species (*Scenario A* established in the EDA). |
| **Information available at inference** | Pre-battle Pokédex statistics: base combat stats (HP, Attack, Defense, Sp. Atk, Sp. Def, Speed), primary and secondary elemental types, legendary indicator, species name (used deterministically to classify special forms), and fighter order. |
| **Excluded features** | `Winner` (*is* the outcome $	o$ direct target leakage); raw IDs `First_pokemon`, `Second_pokemon` (discrete identifiers $	o$ identity memorization; used strictly for group-based partitioning); raw text strings `Name_*` (free text; abstracted into the special form advantage feature); `Generation_*` (verified uninformative, see §8). |

```text
Raw Battle Pairs (First_pokemon, Second_pokemon)
                    ↓
  create_pokemon_features (FunctionTransformer)
  - Merge in-memory with validated Pokédex catalog (stateless lookup)
  - Compute differential features (Speed_diff, Stat_Total_Diff, Atk_Def_Penetration_Diff)
  - Compute domain mechanics (Special_Form_Advantage, Type_Advantage_Ratio)
                    ↓
  ColumnTransformer (Configurable Feature Representations)
  - Numeric Pipeline: StandardScaler (zero-mean or uncentered)
  - Categorical Pipeline: SimpleImputer("None") + OneHotEncoder(min_frequency=0.01)
  - Binary Pipeline: Passthrough (Legendary status)
  - Remainder: "drop" (IDs, names, excluded metadata)
                    ↓
  Model Estimator (Handed off to Assignment 3)
```

> **Test-set limitation note:** The EDA was conducted on the 50,000 historical combats prior to splitting. We acknowledge that the test partition created here was indirectly observed during exploratory analysis (e.g., establishing the 94.05% speed baseline and selecting the 5 master features). However, starting from this notebook, **we make zero decisions using the test partition**: it is isolated and used strictly for structural verification (shape, class balance, and complete absence of pair overlap with train).

# 2. Key Findings from the Exploratory Data Analysis (EDA)

We synthesize the foundational insights from `01_eda_pokemon.ipynb` directly governing our preprocessing decisions (section numbers refer to the EDA notebook):

- **Data Quality & Missing Values (§3.1, §3.2):** Exactly one species (`#63`) lacked a name; its stats match **Primeape** uniquely across all generations. `Type 2` is absent in ~48% of Pokémon; this absence is informative domain signal (*monotype species*), not missing data.
- **Duplicate Combats (§3.3, §11.5):** Exactly 1,952 combats (3.9%) are exact duplicates `(First, Second, Winner)`. 100% of duplicate pairs share the identical winner.
- **Positional Bias & Target Distribution (§4):** First wins 47.20% and Second wins 52.80%, yielding a naive majority baseline of **52.80%**.
- **Speed Dominance & Tie-Breaker Rule (§5.1):** Speed differential has the strongest linear univariate association with the target ($r = +0.678$). When faster, First wins 92.39%; when slower, win rate drops to 4.60%. In the 1,328 exact speed ties (`Speed_diff == 0`), `Second_pokemon` won **100%** of matches across all generations.
- **Combat Antisymmetry (§5.2):** In the 1,822 pairs observed in both battle orientations `(A, B)` and `(B, A)`, 93.96% maintained the same winning species. Combat mechanics are antisymmetric: exchanging fighter order inverts the outcome. Therefore, predictors must be formulated as signed differentials ($X_1 - X_2$).
- **Attribute Differentials & Non-linear Ratios (§6, §10):** Differentials concentrate combat signal. Multiplicative/logarithmic stat ratios were tested and degraded performance ($-0.53$ pp); combat physics in this simulator operate on additive differentials.
- **Elemental Advantage Ratio (§7):** Type advantage provides +10.7 pp win rate improvement. In the Assignment 1 review, the professor requested formal validation of our `TYPE_CHART` against the canonical Gen-6 reference before downstream modeling.
- **Special Forms Dominance (§8):** Mega Evolutions, Primal Reversions, Alternate Forms, and Legendaries win 73.6%–83.3% of battles against standard species.
- **Baseline Benchmarks (§11.1):** Naive Majority Baseline = **52.80%**; Strong Speed-Based Baseline = **94.05%**.

# 3. Translation of EDA Findings to Data Preparation Decisions

Every architectural decision implemented in this pipeline maps directly to an audited EDA finding:

| EDA Finding | Empirical Evidence | Pipeline Decision | Methodological Justification |
|---|---|---|---|
| `Winner` contains combat outcome | Target definition (EDA §1, §11.5) | Compute `Target_First_Wins` and drop `Winner` immediately | Unobserved before battle $	o$ fatal target leakage if retained. |
| High cardinality of species IDs (784 fighters) | `nunique` (EDA §2), win rates (EDA §9) | Exclude raw IDs from features; retain only for grouped split | Forces algorithms to learn generalizable combat mechanics rather than memorizing species keys. |
| Inverted pairs appear in both orientations (1,822 pairs) | Orientation audit (EDA §5.2) | Group train/test partition by **unordered pair** `(min, max)` | Prevents identical matchups (both direct and reversed) from leaking between train and test (*Scenario A* guarantee). |
| 1,952 duplicate battles | `combats.duplicated()` = 1,952 | Retain rows, isolated via unordered pair grouping | `tests.csv` also contains duplicates (70 pairs); grouping isolates all duplicates into the same fold without discarding data. |
| Balanced target (47.2% / 52.8%) | Class distribution (EDA §4) | Verify partition stratification | Preserves the natural positional distribution without synthetic resampling. |
| `Type 2` null in ~48% of species | Domain quality (EDA §3.1) | Impute constant string `"None"`; compute monotype logic natively | Preserves monotype identity as a physical state rather than fabricating categories with mode imputation. |
| Species `#63` missing name | Exact stat match (EDA §3.2) | Deterministic pre-imputation: `Name = "Primeape"` | Verified Pokédex domain truth, not an estimated statistic $	o$ zero leakage. |
| Speed differential dominates | $r = +0.678$ (EDA §6) | Include `Speed_diff` as core differential | Encodes turn initiative, known before combat starts. |
| Total power gap | $r = +0.470$ (EDA §6) | Include `Stat_Total_Diff` | Encodes overall base stat differential without redundant individual totals. |
| Physical damage interaction | Domain mechanics (EDA §6, §10) | Include `Atk_Def_Penetration_Diff` | Non-linear damage ratio $(A_1/D_2 - A_2/D_1)$ capturing physical breakthroughs. |
| Elemental effectiveness | Win rate delta +10.7 pp (EDA §7) | Validate `TYPE_CHART` vs Gen-6 reference; compute `Type_Advantage_Ratio` | Resolves professor's review request; fixes monotype attacker multiplier bug. |
| Special forms dominance | Win rate 73.6%–83.3% (EDA §8) | Compute `Special_Form_Advantage` $\in \{-1, 0, 1\}$ | Captures evolutionary/form gaps as a compact ordinal scale. |
| Categorical nominal types | Heatmaps (EDA §7) | Provide optional `OneHotEncoder(min_frequency=0.01)` | Demonstrates modular nominal handling while grouping rare categories (<1% frequency). |
| Heterogeneous attribute scales | Std dev $pprox 170$ in Total, $pprox 41$ in Speed, $pprox 0.8$ in Type | Standardize via `StandardScaler` inside pipeline | Required for scale-sensitive models (KNN, Logistic Regression, MLP) in Assignment 3. |
| Generation uninformative | $r = +0.015$ with target | Exclude `Generation_*` (empirically confirmed in §8) | Catalogue release index, not an active combat mechanic. |

# 4. Imports, Environment, and Structured Validation Log

We configure reproducible seeds, project paths, and an auditable **validation log** (`validation_log`). Instead of silent assertions, the custom function `check(condition, name, detail)` records every verified structural invariant into an inspectable summary table.

In [1]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "Target_First_Wins"

# Resolve project paths dynamically from root or notebooks/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "pokemon.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from features import (
    TYPE_CHART,
    SPECIAL_FORM_CATEGORIES,
    classify_form,
    calculate_type_multiplier,
)

# Structured validation log
validation_log: list[dict[str, str]] = []


def check(condition: bool, name: str, detail: str) -> None:
    """Validate an invariant: record on success, raise informative ValueError on failure."""
    if not bool(condition):
        raise ValueError(f"Validation FAILED — {name}: {detail}")
    validation_log.append({"Check": name, "Detail": detail, "Status": "PASSED ✓"})


print(f"Python:       {platform.python_version()}")
print(f"pandas:       {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Project root: {PROJECT_ROOT}")


Python:       3.12.3
pandas:       3.0.5
scikit-learn: 1.9.0
Project root: /home/anthonyq/Escritorio/projectMl


# 5. Load and Comprehensive Raw Data Validation

We load the three raw CSV files (`pokemon.csv`, `combats.csv`, `tests.csv`) and enforce strict domain contracts.

### 5.1 Canonical Type Chart Validation (Gen-6 Reference)
In the review of Assignment 1, the professor noted that our manual `TYPE_CHART` must be rigorously verified against the official Generation VI elemental matrix. 
- In Gen-6, there are $18 	imes 18 = 324$ possible attacker-defender type pairs.
- By design, `TYPE_CHART` is a **sparse dictionary** storing only the **120 non-neutral matchups** ($	imes 2.0$, $	imes 0.5$, $	imes 0.0$).
- All remaining 204 combinations evaluate to neutral ($	imes 1.0$) by default in `calculate_type_multiplier`.
Below, we reconstruct the complete official Gen-6 matrix and perform an exhaustive pairwise audit.

In [2]:
POKEMON_PATH = DATA_DIR / "pokemon.csv"
COMBATS_PATH = DATA_DIR / "combats.csv"
TESTS_PATH = DATA_DIR / "tests.csv"

pokemon = pd.read_csv(POKEMON_PATH)
combats = pd.read_csv(COMBATS_PATH)
tests = pd.read_csv(TESTS_PATH)

# --- 1. Schema integrity ---
EXPECTED_POKEMON_COLS = [
    "#", "Name", "Type 1", "Type 2", "HP", "Attack",
    "Defense", "Sp. Atk", "Sp. Def", "Speed", "Generation", "Legendary"
]
EXPECTED_COMBATS_COLS = ["First_pokemon", "Second_pokemon", "Winner"]
EXPECTED_TESTS_COLS = ["First_pokemon", "Second_pokemon"]
STAT_COLS = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

check(set(EXPECTED_POKEMON_COLS).issubset(pokemon.columns), "pokemon.csv schema", f"{pokemon.shape[1]} columns present")
check(set(EXPECTED_COMBATS_COLS).issubset(combats.columns), "combats.csv schema", f"{combats.shape[1]} columns present")
check(set(EXPECTED_TESTS_COLS).issubset(tests.columns), "tests.csv schema", f"{tests.shape[1]} columns present")

# --- 2. Deterministic imputation of Species #63 ---
PRIMEAPE_STATS = {"HP": 65, "Attack": 105, "Defense": 60, "Sp. Atk": 60, "Sp. Def": 70, "Speed": 95}
mask_63 = pokemon["#"] == 63
check(mask_63.sum() == 1, "Species #63 uniqueness", "Exactly 1 record found")
row_63 = pokemon.loc[mask_63].iloc[0]
stat_match_63 = all(row_63[s] == v for s, v in PRIMEAPE_STATS.items())
check(stat_match_63, "Species #63 stat fingerprint", "Matches Primeape base stats exactly")

if pokemon.loc[mask_63, "Name"].isna().any():
    pokemon.loc[mask_63, "Name"] = "Primeape"
check(pokemon["Name"].notna().all(), "Species naming complete", "Zero null names after #63 imputation")

# --- 3. Domain constraints on pokemon ---
check(pokemon["#"].is_unique, "Pokédex ID uniqueness", f"{pokemon['#'].nunique()} unique IDs")
check((pokemon["#"] > 0).all(), "Positive Pokédex IDs", "All IDs > 0")
for s in STAT_COLS:
    check(pokemon[s].notna().all() and (pokemon[s] > 0).all(), f"Stat validity: {s}", "Non-null and strictly positive")

# --- 4. Referential integrity on combats and tests ---
valid_ids = set(pokemon["#"])
check(set(combats["First_pokemon"]).issubset(valid_ids), "Combats First_pokemon valid", "All IDs in Pokédex")
check(set(combats["Second_pokemon"]).issubset(valid_ids), "Combats Second_pokemon valid", "All IDs in Pokédex")
check(set(combats["Winner"]).issubset(valid_ids), "Combats Winner valid", "All IDs in Pokédex")
check((combats["First_pokemon"] != combats["Second_pokemon"]).all(), "No self-battles (combats)", "Zero self-battles")

check(set(tests["First_pokemon"]).issubset(valid_ids), "Tests First_pokemon valid", "All IDs in Pokédex")
check(set(tests["Second_pokemon"]).issubset(valid_ids), "Tests Second_pokemon valid", "All IDs in Pokédex")
check((tests["First_pokemon"] != tests["Second_pokemon"]).all(), "No self-battles (tests)", "Zero self-battles")

valid_winners = (combats["Winner"] == combats["First_pokemon"]) | (combats["Winner"] == combats["Second_pokemon"])
check(valid_winners.all(), "Winner is combatant", "All winners are either First or Second fighter")

# --- 5. Canonical Gen-6 Type Chart Audit ---
GEN6_REFERENCE = {
    "Normal":   ([], ["Rock", "Steel"], ["Ghost"]),
    "Fire":     (["Grass", "Ice", "Bug", "Steel"], ["Fire", "Water", "Rock", "Dragon"], []),
    "Water":    (["Fire", "Ground", "Rock"], ["Water", "Grass", "Dragon"], []),
    "Electric": (["Water", "Flying"], ["Electric", "Grass", "Dragon"], ["Ground"]),
    "Grass":    (["Water", "Ground", "Rock"], ["Fire", "Grass", "Poison", "Flying", "Bug", "Dragon", "Steel"], []),
    "Ice":      (["Grass", "Ground", "Flying", "Dragon"], ["Fire", "Water", "Ice", "Steel"], []),
    "Fighting": (["Normal", "Ice", "Rock", "Dark", "Steel"], ["Poison", "Flying", "Psychic", "Bug", "Fairy"], ["Ghost"]),
    "Poison":   (["Grass", "Fairy"], ["Poison", "Ground", "Rock", "Ghost"], ["Steel"]),
    "Ground":   (["Fire", "Electric", "Poison", "Rock", "Steel"], ["Grass", "Bug"], ["Flying"]),
    "Flying":   (["Grass", "Fighting", "Bug"], ["Electric", "Rock", "Steel"], []),
    "Psychic":  (["Fighting", "Poison"], ["Psychic", "Steel"], ["Dark"]),
    "Bug":      (["Grass", "Psychic", "Dark"], ["Fire", "Fighting", "Poison", "Flying", "Ghost", "Steel", "Fairy"], []),
    "Rock":     (["Fire", "Ice", "Flying", "Bug"], ["Fighting", "Ground", "Steel"], []),
    "Ghost":    (["Psychic", "Ghost"], ["Dark"], ["Normal"]),
    "Dragon":   (["Dragon"], ["Steel"], ["Fairy"]),
    "Dark":     (["Psychic", "Ghost"], ["Fighting", "Dark", "Fairy"], []),
    "Steel":    (["Ice", "Rock", "Fairy"], ["Fire", "Water", "Electric", "Steel"], []),
    "Fairy":    (["Fighting", "Dragon", "Dark"], ["Fire", "Poison", "Steel"], []),
}

reference_chart = {}
for atk, (double, half, zero) in GEN6_REFERENCE.items():
    reference_chart.update({(atk, d): 2.0 for d in double})
    reference_chart.update({(atk, d): 0.5 for d in half})
    reference_chart.update({(atk, d): 0.0 for d in zero})

missing_pairs = sorted(set(reference_chart) - set(TYPE_CHART))
extra_pairs = sorted(set(TYPE_CHART) - set(reference_chart))
diff_values = sorted(k for k in set(reference_chart) & set(TYPE_CHART) if reference_chart[k] != TYPE_CHART[k])

check(len(TYPE_CHART) == 120, "Non-neutral type matchups", "Exactly 120 active entries")
check(not missing_pairs, "Type chart completeness", "0 missing matchups vs Gen-6")
check(not extra_pairs, "Type chart precision", "0 extra matchups vs Gen-6")
check(not diff_values, "Type multiplier accuracy", "All 120 multipliers match Gen-6 canonical reference exactly")
check(set(TYPE_CHART.values()) == {0.0, 0.5, 2.0}, "Type chart valid range", "Only 0.0, 0.5, 2.0 present; 204 neutrals default to 1.0")

# Render validation log
pd.DataFrame(validation_log)


,Check,Detail,Status
0,pokemon.csv schema,12 columns present,PASSED ✓
1,combats.csv schema,3 columns present,PASSED ✓
2,tests.csv schema,2 columns present,PASSED ✓
3,Species #63 uniqueness,Exactly 1 record found,PASSED ✓
4,Species #63 stat fingerprint,Matches Primeape base stats exactly,PASSED ✓
5,Species naming complete,Zero null names after #63 imputation,PASSED ✓
6,Pokédex ID uniqueness,800 unique IDs,PASSED ✓
7,Positive Pokédex IDs,All IDs > 0,PASSED ✓
8,Stat validity: HP,Non-null and strictly positive,PASSED ✓
9,Stat validity: Attack,Non-null and strictly positive,PASSED ✓


# 6. Target Isolation and Leakage Prevention

We extract the binary target `Target_First_Wins` and **immediately isolate `Winner`**. The raw `Winner` column contains the ID of the victorious Pokémon: retaining it in the feature matrix $X$ would constitute fatal target leakage.

The input feature matrix $X$ contains **strictly the combatant identifiers** `First_pokemon` and `Second_pokemon`. All stats, types, and interactions will be derived downstream inside the pipeline via our stateless transformer.

In [3]:
y = (combats["Winner"] == combats["First_pokemon"]).astype(int)
y.name = TARGET_COLUMN

# Feature matrix X contains strictly the combatant pair IDs
X = combats[["First_pokemon", "Second_pokemon"]].copy()

target_summary = (
    y.value_counts(normalize=True)
    .sort_index()
    .rename("Proportion")
    .to_frame()
)
target_summary["Count"] = y.value_counts().sort_index()
target_summary.index = ["Second wins (0)", "First wins (1)"]

print(f"Dataset total battles: {len(X):,}")
target_summary


Dataset total battles: 50,000


,Proportion,Count
Second wins (0),0.52798,26399
First wins (1),0.47202,23601


# 7. Grouped Train/Test Partitioning by Unordered Battle Pair

### Methodological Justification: Why a naive split causes pair leakage
In standard tabular datasets (such as Titanic), each row represents an independent entity. In Pokémon, rows represent pairwise interactions:
- **1,952 exact duplicates** exist in `combats.csv`.
- **1,822 pairs** appear in both orientations `(A, B)` and `(B, A)`.
- If an unconstrained split is applied, **~11.8%** of test battles have their unordered matchup present in training data.

We define the group key as the **unordered pair string** `f"{min(A, B)}_{max(A, B)}"`. Partitioning via `GroupShuffleSplit` guarantees that every combat involving fighters $\{A, B\}$ (regardless of orientation or repetition) falls strictly on one side of the partition. This enforces true **Scenario A** evaluation (*new head-to-head matchup between known Pokémon*).

In [4]:
# Construct unordered pair group key
groups = pd.Series(
    [f"{min(a, b)}_{max(a, b)}" for a, b in zip(X["First_pokemon"], X["Second_pokemon"])],
    index=X.index,
    name="pair_group",
)

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Audit pair isolation and class balance
train_pairs = set(groups.iloc[train_idx])
test_pairs = set(groups.iloc[test_idx])
check(train_pairs.isdisjoint(test_pairs), "Pair leakage prevention", "Zero overlapping pairs between train and test")

train_bal = y_train.value_counts(normalize=True).sort_index()
test_bal = y_test.value_counts(normalize=True).sort_index()

split_report = pd.DataFrame({
    "Partition": ["Training (Train)", "Holdout (Test)"],
    "Rows": [len(X_train), len(X_test)],
    "Proportion": [len(X_train) / len(X), len(X_test) / len(X)],
    "Unique Pairs": [len(train_pairs), len(test_pairs)],
    "Class 0 (Second Wins)": [f"{train_bal.iloc[0]:.4%}", f"{test_bal.iloc[0]:.4%}"],
    "Class 1 (First Wins)": [f"{train_bal.iloc[1]:.4%}", f"{test_bal.iloc[1]:.4%}"],
})

display(split_report)
print(f"Verified: 0 overlapping pairs across {len(train_pairs) + len(test_pairs):,} total unique matchups.")


,Partition,Rows,Proportion,Unique Pairs,Class 0 (Second Wins),Class 1 (First Wins)
0,Training (Train),40078,0.80156,36980,52.8295%,47.1705%
1,Holdout (Test),9922,0.19844,9246,52.6708%,47.3292%


Verified: 0 overlapping pairs across 46,226 total unique matchups.


# 8. Stateless Feature Engineering Function

We build a **stateless** feature extraction function encapsulated inside a closure: `make_pokemon_feature_creator(catalog)`.
1. **Catalog closure:** The cleaned Pokédex catalog is stored as an immutable in-memory lookup table. The function performs an in-memory vector join in ~0.05s, completely decoupling the pipeline from disk I/O.
2. **Index preservation:** `pd.merge` naturally resets non-contiguous indices (like those from `GroupShuffleSplit`). Our implementation explicitly preserves and restores the original DataFrame index, preventing silent misalignment with target vectors.
3. **Five Domain Master Features:**
   - `Speed_diff` = $	ext{Speed}_1 - 	ext{Speed}_2$
   - `Stat_Total_Diff` = $\sum 	ext{Stats}_1 - \sum 	ext{Stats}_2$
   - `Atk_Def_Penetration_Diff` = $rac{	ext{Attack}_1}{\max(	ext{Defense}_2, 1)} - rac{	ext{Attack}_2}{\max(	ext{Defense}_1, 1)}$
   - `Special_Form_Advantage` = $	ext{Is\_Special}_1 - 	ext{Is\_Special}_2 \in \{-1, 0, +1\}$
   - `Type_Advantage_Ratio` = $\log_2\left(rac{	ext{Eff}_{1 	o 2} + 0.1}{	ext{Eff}_{2 	o 1} + 0.1}ight)$ with corrected monotype multiplier logic.

In [5]:
def make_pokemon_feature_creator(pokemon_catalog: pd.DataFrame):
    """
    Factory returning a stateless transformer that enriches raw combat pairs
    with Pokédex attributes and calculates the 5 domain master features.
    """
    cat = pokemon_catalog.copy().rename(columns={"#": "id"})

    # Pre-classify evolutionary/combat forms on the static catalog
    cat["form"] = [classify_form(n, l) for n, l in zip(cat["Name"], cat["Legendary"])]
    merge_cols = [c for c in cat.columns if c != "id"]
    stat_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

    def create_pokemon_features(battles: pd.DataFrame) -> pd.DataFrame:
        result = battles.copy()
        original_index = result.index.copy()

        # Merge Position 1 fighter
        first = cat.rename(columns={c: f"{c}_first" for c in merge_cols})
        result = result.merge(
            first, left_on="First_pokemon", right_on="id", how="left"
        ).drop(columns=["id"], errors="ignore")

        # Merge Position 2 fighter
        second = cat.rename(columns={c: f"{c}_second" for c in merge_cols})
        result = result.merge(
            second, left_on="Second_pokemon", right_on="id", how="left"
        ).drop(columns=["id"], errors="ignore")

        # Explicitly restore original DataFrame index to prevent merge misalignment
        result.index = original_index

        # 1. Speed Differential
        result["Speed_diff"] = result["Speed_first"] - result["Speed_second"]

        # 2. Total Base Stat Differential
        total_first = sum(result[f"{s}_first"] for s in stat_cols)
        total_second = sum(result[f"{s}_second"] for s in stat_cols)
        result["Stat_Total_Diff"] = total_first - total_second

        # 3. Physical Attack/Defense Penetration Differential
        result["Atk_Def_Penetration_Diff"] = (
            result["Attack_first"] / result["Defense_second"].replace(0, 1)
            - result["Attack_second"] / result["Defense_first"].replace(0, 1)
        )

        # 4. Special Form Advantage (Mega, Primal, Legendary, Alt. Form)
        is_sp_1 = result["form_first"].isin(SPECIAL_FORM_CATEGORIES).astype(int)
        is_sp_2 = result["form_second"].isin(SPECIAL_FORM_CATEGORIES).astype(int)
        result["Special_Form_Advantage"] = is_sp_1 - is_sp_2

        # 5. Type Advantage Ratio (Corrected for Monotype Attackers)
        eff_1v2 = pd.Series(
            [
                calculate_type_multiplier(t1_1, t2_1, t1_2, t2_2)
                for t1_1, t2_1, t1_2, t2_2 in zip(
                    result["Type 1_first"], result["Type 2_first"],
                    result["Type 1_second"], result["Type 2_second"],
                )
            ],
            index=result.index,
        )
        eff_2v1 = pd.Series(
            [
                calculate_type_multiplier(t1_2, t2_2, t1_1, t2_1)
                for t1_1, t2_1, t1_2, t2_2 in zip(
                    result["Type 1_first"], result["Type 2_first"],
                    result["Type 1_second"], result["Type 2_second"],
                )
            ],
            index=result.index,
        )
        result["Type_Advantage_Ratio"] = np.log2((eff_1v2 + 0.1) / (eff_2v1 + 0.1))

        return result

    return create_pokemon_features


create_pokemon_features = make_pokemon_feature_creator(pokemon)

# Preview transformation on training slice
preview = create_pokemon_features(X_train.head(5))
preview_cols = [
    "First_pokemon", "Second_pokemon", "Name_first", "Name_second",
    "Speed_diff", "Stat_Total_Diff", "Atk_Def_Penetration_Diff",
    "Special_Form_Advantage", "Type_Advantage_Ratio"
]
display(preview[preview_cols])


,First_pokemon,Second_pokemon,Name_first,Name_second,Speed_diff,Stat_Total_Diff,Atk_Def_Penetration_Diff,Special_Form_Advantage,Type_Advantage_Ratio
0,266,298,Larvitar,Nuzleaf,-19,-40,0.200000,0,-1.898120
1,702,701,Virizion,Terrakion,0,0,-0.791667,0,0.932886
2,191,668,Togetic,Beheeyem,0,-80,-0.349020,0,0.000000
3,237,683,Slugma,Druddigon,-28,-235,-2.555556,0,-0.874469
4,151,231,Omastar,Shuckle,50,-10,0.180870,0,0.932886


# 9. Column Roles, Generation Audit, and Multicollinearity Analysis

Following the pedagogical principle from the Titanic template, every single column generated by our feature extraction must be assigned an **auditable analytical role**.

### 9.1 Empirical Confirmation: Discarding Generation
We test whether generation differences carry predictive signal using strictly training data. The correlation with the target is near zero ($r pprox +0.015$), confirming that Pokédex release generation is not a combat mechanic.

### 9.2 Exact Linear Redundancy (Multicollinearity) Audit
The assignment guidelines mandate avoiding unprincipled redundant representations:
- `Speed_diff` $\equiv 	ext{Speed}_1 - 	ext{Speed}_2$ (exact linear combination).
- `Stat_Total_Diff` $\equiv \sum 	ext{Stats}_1 - \sum 	ext{Stats}_2$ (exact linear combination).
- `Atk_Def_Penetration_Diff` is non-linear (not redundant).
**Modeling implication:** In tree-based models (Random Forest, Gradient Boosting), explicit differences provide efficient orthogonal partition thresholds. In unregularized linear models, Arm C induces **perfect multicollinearity**; this theoretical finding justifies why linear models must be regularized or evaluated on Arm A/B.

In [6]:
train_features = create_pokemon_features(X_train)

# --- 1. Generation difference audit (Train set only) ---
gen_diff = (train_features["Generation_first"] - train_features["Generation_second"]).rename("Generation_diff")
gen_corr = gen_diff.corr(y_train)
print(f"Generation difference correlation with target (train): {gen_corr:+.4f}")
check(abs(gen_corr) < 0.05, "Generation uninformative", f"Correlation {gen_corr:+.4f} is negligible (< 0.05)")

# --- 2. Comprehensive column role mapping ---
RAW_NUMERIC = [f"{stat}_{side}" for stat in STAT_COLS for side in ("first", "second")]
ENGINEERED_NUMERIC = ["Speed_diff", "Stat_Total_Diff", "Atk_Def_Penetration_Diff", "Type_Advantage_Ratio"]
NOMINAL = ["Type 1_first", "Type 2_first", "Type 1_second", "Type 2_second"]
ORDINAL = ["Special_Form_Advantage"]
BINARY = ["Legendary_first", "Legendary_second"]
EXCLUDED = {
    "First_pokemon": "Fighter ID -> memorization hazard; used only for grouped partitioning",
    "Second_pokemon": "Fighter ID -> memorization hazard; used only for grouped partitioning",
    "Name_first": "Raw text; abstracted into Special_Form_Advantage",
    "Name_second": "Raw text; abstracted into Special_Form_Advantage",
    "form_first": "Internal categorical helper; abstracted into Special_Form_Advantage",
    "form_second": "Internal categorical helper; abstracted into Special_Form_Advantage",
    "Generation_first": "Empirically uninformative (r = +0.015)",
    "Generation_second": "Empirically uninformative (r = +0.015)",
}

roles_list = (
    [(c, "Raw Numeric Stat") for c in RAW_NUMERIC]
    + [(c, "Engineered Numeric (Differential)") for c in ENGINEERED_NUMERIC]
    + [(c, "Nominal Categorical (Types)") for c in NOMINAL]
    + [(c, "Ordinal Feature (Special Forms)") for c in ORDINAL]
    + [(c, "Binary Indicator") for c in BINARY]
    + [(c, f"Excluded: {why}") for c, why in EXCLUDED.items()]
)
roles_df = pd.DataFrame(roles_list, columns=["Column", "Assigned Role"])

# Coverage assertion: every column in train_features must have an assigned role
all_cols = set(train_features.columns)
role_cols = set(roles_df["Column"])
check(all_cols == role_cols, "100% Column Role Coverage", f"Unassigned columns: {sorted(all_cols - role_cols)}")

# --- 3. Exact Linear Redundancy Audit ---
redundancy_audit = pd.DataFrame({
    "Feature": ["Speed_diff", "Stat_Total_Diff", "Atk_Def_Penetration_Diff"],
    "Reconstruction Formula": [
        "Speed_first - Speed_second",
        "sum(Stats_first) - sum(Stats_second)",
        "(Atk_1/Def_2) - (Atk_2/Def_1) [Non-linear quotient]",
    ],
    "Exact Linear Combination": [
        np.array_equal(train_features["Speed_diff"], train_features["Speed_first"] - train_features["Speed_second"]),
        np.array_equal(
            train_features["Stat_Total_Diff"],
            train_features[[f"{s}_first" for s in STAT_COLS]].sum(axis=1) - train_features[[f"{s}_second" for s in STAT_COLS]].sum(axis=1)
        ),
        False,
    ],
})

display(redundancy_audit)
display(roles_df.groupby("Assigned Role", sort=False)["Column"].apply(list).to_frame())


Generation difference correlation with target (train): +0.0163


,Feature,Reconstruction Formula,Exact Linear Combination
0,Speed_diff,Speed_first - Speed_second,True
1,Stat_Total_Diff,sum(Stats_first) - sum(Stats_second),True
2,Atk_Def_Penetration_Diff,(Atk_1/Def_2) - (Atk_2/Def_1) [Non-linear quot...,False


,Column
Assigned Role,
Raw Numeric Stat,"[HP_first, HP_second, Attack_first, Attack_sec..."
Engineered Numeric (Differential),"[Speed_diff, Stat_Total_Diff, Atk_Def_Penetrat..."
Nominal Categorical (Types),"[Type 1_first, Type 2_first, Type 1_second, Ty..."
Ordinal Feature (Special Forms),[Special_Form_Advantage]
Binary Indicator,"[Legendary_first, Legendary_second]"
Excluded: Fighter ID -> memorization hazard; used only for grouped partitioning,"[First_pokemon, Second_pokemon]"
Excluded: Raw text; abstracted into Special_Form_Advantage,"[Name_first, Name_second]"
Excluded: Internal categorical helper; abstracted into Special_Form_Advantage,"[form_first, form_second]"
Excluded: Empirically uninformative (r = +0.015),"[Generation_first, Generation_second]"


# 10. Sub-Pipelines and Multi-Arm Specification

We structure variable transformations into dedicated sub-pipelines combined via `ColumnTransformer`:
- **Numeric variables:** Scaled with `StandardScaler()`.
- **Nominal variables (Types):** Missing `Type 2` values are imputed as `"None"` (representing monotype status). We then apply `OneHotEncoder(min_frequency=0.01, handle_unknown="infrequent_if_exist")`. Categories appearing in less than 1% of training battles (such as primary `Flying` or secondary `Bug`/`Electric`) are grouped into an `infrequent` bin, stabilizing estimation.
- **Binary/Ordinal variables:** Passed through directly (`"passthrough"`).

### Formal Representation Arms (Roadmap from Assignment 1)
We define four explicit feature representations:
- **`A_raw`:** 12 raw base stats.
- **`B_engineered`:** 5 master domain features (4 numeric differentials + 1 ordinal form advantage).
- **`C_combined`:** 17 features (Raw + Engineered) — our **primary baseline**.
- **`full`:** 89 features (Combined + One-Hot Encoded Types + Legendary indicators).

In [7]:
# Sub-pipelines
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(min_frequency=0.01, handle_unknown="infrequent_if_exist", sparse_output=False)),
])

# Multi-arm representation dictionary
FEATURE_SETS = {
    "A_raw": {
        "numeric": RAW_NUMERIC,
    },
    "B_engineered": {
        "numeric": ENGINEERED_NUMERIC,
        "passthrough": ORDINAL,
    },
    "C_combined": {
        "numeric": RAW_NUMERIC + ENGINEERED_NUMERIC,
        "passthrough": ORDINAL,
    },
    "full": {
        "numeric": RAW_NUMERIC + ENGINEERED_NUMERIC,
        "nominal": NOMINAL,
        "passthrough": ORDINAL + BINARY,
    },
}


def make_preprocessor(feature_set: str = "C_combined") -> ColumnTransformer:
    """Construct a ColumnTransformer tailored to the specified representation arm."""
    spec = FEATURE_SETS[feature_set]
    transformers = []
    if "numeric" in spec:
        transformers.append(("numeric", numeric_pipeline, spec["numeric"]))
    if "nominal" in spec:
        transformers.append(("nominal", nominal_pipeline, spec["nominal"]))
    if "passthrough" in spec:
        transformers.append(("passthrough", "passthrough", spec["passthrough"]))

    return ColumnTransformer(transformers=transformers, remainder="drop", verbose_feature_names_out=False)


# Default preprocessor represents Arm C (Combined - 17 features)
preprocessor = make_preprocessor("C_combined")

arm_summary = []
for name, spec in FEATURE_SETS.items():
    prep_test = make_preprocessor(name)
    sample_out = prep_test.fit_transform(train_features)
    arm_summary.append({
        "Representation Arm": name,
        "Output Dimension (Columns)": sample_out.shape[1],
        "Sub-Pipelines Included": ", ".join(spec.keys()),
    })

pd.DataFrame(arm_summary)


,Representation Arm,Output Dimension (Columns),Sub-Pipelines Included
0,A_raw,12,numeric
1,B_engineered,5,"numeric, passthrough"
2,C_combined,17,"numeric, passthrough"
3,full,89,"numeric, nominal, passthrough"


# 11. End-to-End Preparation Pipeline and Structural Checks

We assemble the complete preparation pipeline:
$$	ext{Raw Battle Pairs } X \longrightarrow 	ext{FunctionTransformer(create\_pokemon\_features)} \longrightarrow 	ext{ColumnTransformer(preprocessor)} \longrightarrow 	ext{Prepared Matrix}$$

We fit the pipeline exclusively on `X_train` and run exhaustive structural and leakage assertions.

In [8]:
preparation_pipeline = Pipeline([
    ("feature_engineering", FunctionTransformer(create_pokemon_features, validate=False)),
    ("preprocessor", preprocessor),
])

X_train_prepared = preparation_pipeline.fit_transform(X_train)
out_feature_names = preparation_pipeline.named_steps["preprocessor"].get_feature_names_out()

# --- Structural checks ---
check(X_train_prepared.shape[0] == len(X_train), "Row count preservation", f"{X_train_prepared.shape[0]:,} rows preserved")
check(X_train_prepared.shape[1] == 17, "Arm C feature count", f"{X_train_prepared.shape[1]} features in output")
check(np.isfinite(X_train_prepared).all(), "Finite numerical values", "Zero NaN, null, or infinite values")

# --- Strict Leakage Checks ---
leaked_cols = [c for c in [*EXCLUDED.keys(), "Winner", TARGET_COLUMN] if c in out_feature_names]
check(not leaked_cols, "Leakage prevention (Target/IDs/Excluded)", f"Leaked columns: {leaked_cols or 'None ✓'}")

print(f"Raw training input:    {X_train.shape} (strictly First_pokemon, Second_pokemon)")
print(f"Prepared output matrix: {X_train_prepared.shape}")
print(f"All structural and leakage assertions passed successfully.")

pd.DataFrame({"Feature Name": out_feature_names, "Index": range(len(out_feature_names))})


Raw training input:    (40078, 2) (strictly First_pokemon, Second_pokemon)
Prepared output matrix: (40078, 17)
All structural and leakage assertions passed successfully.


,Feature Name,Index
0,HP_first,0
1,HP_second,1
2,Attack_first,2
3,Attack_second,3
4,Defense_first,4
5,Defense_second,5
6,Sp. Atk_first,6
7,Sp. Atk_second,7
8,Sp. Def_first,8
9,Sp. Def_second,9


# 12. End-to-End Model Pipeline Factory

The modeling notebook (Assignment 3) will evaluate candidate estimators. Using `clone()`, `build_model_pipeline(estimator, feature_set)` creates an unfitted, isolated pipeline instance for any estimator and representation arm. This guarantees that different cross-validation folds and candidate models share zero learned state (such as scaling parameters).

In [9]:
def build_model_pipeline(estimator, feature_set: str = "C_combined") -> Pipeline:
    """
    Build an unfitted, end-to-end pipeline ready for cross-validation or training.
    Accepts raw battle pairs (First_pokemon, Second_pokemon) and returns class predictions.
    """
    prep = make_preprocessor(feature_set)
    return Pipeline([
        ("feature_engineering", FunctionTransformer(create_pokemon_features, validate=False)),
        ("preprocessor", clone(prep)),
        ("model", estimator),
    ])


# Verification of independent state per pipeline instance
from sklearn.tree import DecisionTreeClassifier

pipe_1 = build_model_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE), "A_raw")
pipe_2 = build_model_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE), "C_combined")

check(
    pipe_1.named_steps["preprocessor"] is not pipe_2.named_steps["preprocessor"],
    "Pipeline isolation",
    "Cloned preprocessors do not share memory or learned state",
)
print("Model pipeline factory verified successfully. ✓")


Model pipeline factory verified successfully. ✓


# 13. Pipeline Verification on `tests.csv`

A critical architectural test: the **exact same pipeline object** must transform both `combats.csv` (training) and `tests.csv` (the unlabelled evaluation set). Since `tests.csv` has no `Winner` column, pipelines that perform external joins outside `FunctionTransformer` fail immediately.

Here, we pass `tests.csv` through `preparation_pipeline.transform()` directly.

In [10]:
tests_input = tests[["First_pokemon", "Second_pokemon"]].copy()
tests_prepared = preparation_pipeline.transform(tests_input)

check(tests_prepared.shape[0] == len(tests), "tests.csv row preservation", f"{tests_prepared.shape[0]:,} rows transformed")
check(tests_prepared.shape[1] == 17, "tests.csv feature alignment", f"{tests_prepared.shape[1]} features generated")
check(np.isfinite(tests_prepared).all(), "tests.csv finite values", "Zero NaNs in transformed test matrix")

print(f"tests.csv transformed successfully: {tests_prepared.shape}")
print("Full end-to-end compatibility between training and test workflows verified. ✓")


tests.csv transformed successfully: (10000, 17)
Full end-to-end compatibility between training and test workflows verified. ✓


# 14. Feature Signal Quality, Artifacts, and Handoff to Modeling

### 14.1 Univariate Correlation Audit (Training Set Only)
We inspect the point-biserial correlation of each feature with the target strictly on the training partition ($X_{\text{train}}, y_{\text{train}}$).

In [11]:
# Point-biserial correlation table on training set
correlations = (
    train_features[RAW_NUMERIC + ENGINEERED_NUMERIC + ORDINAL]
    .corrwith(y_train)
    .rename("r_with_target")
    .to_frame()
)
correlations["abs_r"] = correlations["r_with_target"].abs()
correlations = correlations.sort_values("abs_r", ascending=False)

display(correlations)


,r_with_target,abs_r
Speed_diff,0.680450,0.680450
Speed_first,0.493619,0.493619
Stat_Total_Diff,0.471021,0.471021
Speed_second,-0.467856,0.467856
Special_Form_Advantage,0.292939,0.292939
Attack_first,0.261369,0.261369
Sp. Atk_first,0.251853,0.251853
Attack_second,-0.250510,0.250510
Atk_Def_Penetration_Diff,0.246160,0.246160
Sp. Atk_second,-0.242975,0.242975


### 14.2 Serialization of Split Manifest and Reproducibility Metadata
Following the template architecture, we serialize:
1. `artifacts/split_assignments_pokemon.csv`: Contains `combat_id` (0..49,999 index), `First_pokemon`, `Second_pokemon`, and the assigned `split` (`train` or `test`).
2. `artifacts/feature_engineering_metadata.json`: Contains reproducibility metadata, library versions, feature dictionaries, and audit notes.

In [12]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Split manifest with unambiguous combat_id integer index
train_manifest = X_train.copy()
train_manifest["split"] = "train"

test_manifest = X_test.copy()
test_manifest["split"] = "test"

split_manifest = pd.concat([train_manifest, test_manifest], ignore_index=False).sort_index()
split_manifest.index.name = "combat_id"

manifest_path = ARTIFACTS_DIR / "split_assignments_pokemon.csv"
split_manifest.to_csv(manifest_path)
print(f"Split manifest saved: {len(split_manifest):,} records → {manifest_path}")

# 2. Reproducibility metadata
metadata = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "split_method": "GroupShuffleSplit by unordered pair (min(id), max(id))",
    "n_train": len(X_train),
    "n_test": len(X_test),
    "class_balance_train": {int(k): float(v) for k, v in train_bal.items()},
    "class_balance_test": {int(k): float(v) for k, v in test_bal.items()},
    "feature_sets": {name: list(spec.keys()) for name, spec in FEATURE_SETS.items()},
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "type_chart_audit": {
        "non_neutral_matchups": len(TYPE_CHART),
        "canonical_reference": "Gen-6 18x18 type chart verified (120 active, 204 neutral 1.0)",
        "missing_matchups": len(missing_pairs),
        "extra_matchups": len(extra_pairs),
    },
    "test_set_protocol": (
        "Holdout test partition is strictly locked. No model tuning or threshold selection "
        "may inspect test data prior to final submission."
    ),
}

metadata_path = ARTIFACTS_DIR / "feature_engineering_metadata.json"
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Metadata saved successfully → {metadata_path}")


Split manifest saved: 50,000 records → /home/anthonyq/Escritorio/projectMl/artifacts/split_assignments_pokemon.csv
Metadata saved successfully → /home/anthonyq/Escritorio/projectMl/artifacts/feature_engineering_metadata.json


### 14.3 Handoff to Modeling (Assignment 3 Protocol)

The upcoming modeling notebook (`03_model_evaluation_pokemon.ipynb`) should adhere to the following experimental guidelines:

1. **Partition Loading:** Load partition assignments from `artifacts/split_assignments_pokemon.csv` via `combat_id` to guarantee identical folds.
2. **Cross-Validation Scheme:** Cross-validation within the training partition must use `GroupKFold` or `StratifiedGroupKFold` on the unordered pair `(min(id), max(id))` to preserve zero pair leakage across folds.
3. **Multi-Arm Benchmarking:** Compare the three formal representations (`A_raw`, `B_engineered`, `C_combined`, and optionally `full`) using `build_model_pipeline(estimator, feature_set)`.
4. **Benchmarks to Surpass:**
   - **Naive Majority Baseline:** **52.80%** accuracy.
   - **Strong Speed-Based Baseline:** **94.05%** accuracy ($Speed_1 > Speed_2 \implies 1$; $Speed_1 < Speed_2 \implies 0$; $Speed_1 = Speed_2 \implies 0$).
   - **Target Performance Goal:** > **95.30%** accuracy on unseen test matchups.
5. **Algorithmic Scope:**
   - *Linear Models (Logistic Regression):* Use as a weak baseline (~88% accuracy due to non-linear speed mechanics and multicollinearity in Arm C).
   - *Ensemble Decision Trees (Random Forest, Gradient Boosting / LightGBM):* Primary candidates designed to exploit orthogonal differential thresholds and resolve the ~5.95% underdog regime.